In [81]:
import carla
import random #to pick random spawn point
import cv2
import numpy as np #for reshaping images
import time
import networkx as nx
import sys
import math
sys.path.append('/home/g02-f26/carla0916/PythonAPI/carla') # global route planner 
from agents.navigation.global_route_planner import GlobalRoutePlanner 

In [82]:
client = carla.Client('localhost', 2000)

In [83]:
# Define environment/world and get possible places to spawn a car
world = client.get_world()
spawn_points = world.get_map().get_spawn_points()

In [84]:


# Look for blueprint of a mini car
vehicle_bp = world.get_blueprint_library().filter('*mini*')

if not vehicle_bp:
    print("Error: No 'mini' vehicle blueprint found!")
else:
    blueprint = vehicle_bp[0]
    vehicle = None

    # Loop through spawn points until one successfully spawns without crashing
    for start_point in spawn_points:
        try:
            vehicle = world.try_spawn_actor(blueprint, start_point)
            if vehicle is not None:
                print(f"Successfully spawned vehicle at {start_point.location}")
                break # Exit loop once successfully spawned
        except Exception as e:
            print(f"Failed to spawn at this point, trying next one... Error: {e}")

    if vehicle is None:
        print("Error: Could not spawn the vehicle at any available spawn points.")


Successfully spawned vehicle at Location(x=-64.644844, y=24.471010, z=0.600000)


In [85]:
#camera initialization
CAMERA_POS_Z = 1.6 #UP/DOWN
CAMERA_POS_X = 0.9 #FORWARD/BACKWARDS

camera_bp = world.get_blueprint_library().find('sensor.camera.rgb')
camera_bp.set_attribute('image_size_x', '640') #need to stick to this proportion/aspcet ration or else will not work
camera_bp.set_attribute('image_size_y', '360')

camera_init_trans = carla.Transform(carla.Location(z=CAMERA_POS_Z, x=CAMERA_POS_X))
camera = world.spawn_actor(camera_bp, camera_init_trans, attach_to=vehicle)

#def camera_callback(image,data_dict):
 #   data_dict['image'] = np.reshape(np.copy(image.raw_data), (image.height, image.width, 4))
    
image_w = camera_bp.get_attribute('image_size_x').as_int()
image_h = camera_bp.get_attribute('image_size_y').as_int()
#camera_data = {'image': np.zeros((image_h, image_w, 4))}
#this opens camera livestream
#camera.listen(lambda image: camera_callback(image,camera_data))

In [86]:
#imu/GPS initialization
gnss_bp = world.get_blueprint_library().find('sensor.other.gnss')
gnss = world.spawn_actor(gnss_bp, carla.Transform(), attach_to=vehicle)

imu_bp = world.get_blueprint_library().find('sensor.other.imu')
imu = world.spawn_actor(imu_bp, carla.Transform(), attach_to=vehicle)
"""
def gnss_callback(data, data_dict):
    data_dict['gnss'] = [data.latitude, data.longitude]

def imu_callback(data, data_dict):
    data_dict['imu'] = {
        'gyro' : data.gyroscope,
        'accel' : data.accelerometer,
        'compass' : data.compass
    }
"""

"\ndef gnss_callback(data, data_dict):\n    data_dict['gnss'] = [data.latitude, data.longitude]\n\ndef imu_callback(data, data_dict):\n    data_dict['imu'] = {\n        'gyro' : data.gyroscope,\n        'accel' : data.accelerometer,\n        'compass' : data.compass\n    }\n"

In [87]:
#fake speed
class SpeedCoordinator:
    def __init__(self, true_speed_fn, fake_speed_ratio, ramp_duration_s, smoothing=0.2):
        self.true_speed_fn = true_speed_fn
        self.target_ratio = fake_speed_ratio
        self.ramp_duration = ramp_duration_s
        self.t0 = None
        self._last_t = None
        self._last_error = None
        self.smoothing = smoothing      # lower = smoother/slower to react
        self._smoothed_speed = None

    def get_smoothed_speed(self):
        raw = self.true_speed_fn()
        if self._smoothed_speed is None:
            self._smoothed_speed = raw
        else:
            self._smoothed_speed += self.smoothing * (raw - self._smoothed_speed)
        return self._smoothed_speed

    def current_ratio(self, t):
        if self.t0 is None:
            self.t0 = t
        elapsed = t - self.t0
        frac = min(elapsed / self.ramp_duration, 1.0)
        return 1.0 + (self.target_ratio - 1.0) * frac

    def fake_speed(self, t):
        return self.get_smoothed_speed() * self.current_ratio(t)

    def extra_accel(self, t):
        true_speed = self.get_smoothed_speed()
        error = self.fake_speed(t) - true_speed
        if self._last_t is None:
            self._last_t, self._last_error = t, error
            return 0.0
        dt = t - self._last_t
        prev_error = self._last_error
        self._last_t, self._last_error = t, error
        return (error - prev_error) / dt if dt > 0 else 0.0

speed_coord = SpeedCoordinator(
    true_speed_fn=lambda: vehicle.get_velocity().length(),
    fake_speed_ratio=1.4,
    ramp_duration_s=6.0
)

In [88]:
class FakePositionTracker:
    """Accumulates extra displacement (in meters, N/E) caused by the fake-vs-true speed gap."""
    def __init__(self, speed_coordinator):
        self.speed_coord = speed_coordinator
        self.extra_north_m = 0.0
        self.extra_east_m = 0.0
        self._last_t = None

    def update(self, t, true_speed, heading_deg):
        if self._last_t is None:
            self._last_t = t
            return self.extra_north_m, self.extra_east_m

        dt = t - self._last_t
        self._last_t = t

        fake_speed = self.speed_coord.fake_speed(t)
        extra_speed = fake_speed - true_speed  # m/s of "phantom" motion

        heading_rad = math.radians(heading_deg)
        # CARLA yaw=0 faces world +X; verify this mapping against your map's
        # georeference before trusting N/E signs
        self.extra_north_m += extra_speed * math.cos(heading_rad) * dt
        self.extra_east_m  += extra_speed * math.sin(heading_rad) * dt
        return self.extra_north_m, self.extra_east_m


def apply_gps_position_bias(gnss_data, extra_north_m, extra_east_m):
    R = 6378137.0  # Earth radius, meters
    lat_rad = math.radians(gnss_data.latitude)

    spoofed_lat = gnss_data.latitude + (extra_north_m / R) * (180 / math.pi)
    spoofed_lon = gnss_data.longitude + (extra_east_m / (R * math.cos(lat_rad))) * (180 / math.pi)

    return {
        'latitude': spoofed_lat,
        'longitude': spoofed_lon,
        'original_latitude': gnss_data.latitude,
        'original_longitude': gnss_data.longitude,
    }

pos_tracker = FakePositionTracker(speed_coord)

In [89]:


def apply_imu_speed_bias(imu_data, accel_bias):
    accel = imu_data.accelerometer
    # CARLA IMU local frame: x = vehicle forward
    biased_accel = carla.Vector3D(accel.x + accel_bias, accel.y, accel.z)
    return {
        'gyro': imu_data.gyroscope,
        'accelerometer': biased_accel,
        'original_accel': accel,
        'compass': imu_data.compass,
    }

In [90]:
#warp camera image to immitate speeding up 
class OpticalFlowSpoofer:
    def __init__(self, speed_coordinator, expansion_gain=20.0, max_scale=2.0):
        self.speed_coord = speed_coordinator
        self.expansion_gain = expansion_gain
        self.max_scale = max_scale  # safety clamp

    def apply(self, frame, t, dt, focus_of_expansion=None):
        h, w, _ = frame.shape
        if focus_of_expansion is None:
            focus_of_expansion = (w // 2, int(h * 0.45))

        ratio = self.speed_coord.current_ratio(t)
        # bounded, per-frame scale — no persistent state
        scale = 1.0 + (ratio - 1.0) * self.expansion_gain * dt
        scale = min(scale, self.max_scale)

        M = cv2.getRotationMatrix2D(focus_of_expansion, 0, scale)
        warped = cv2.warpAffine(frame, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
        return warped

flow_spoofer = OpticalFlowSpoofer(speed_coord)

In [91]:
def process_image(image):
    array = np.frombuffer(image.raw_data, dtype=np.uint8)

    array = array.reshape(
        (image.height, image.width, 4)
    )

    bgr = cv2.cvtColor(array, cv2.COLOR_BGRA2BGR)

    return bgr

In [92]:
#callbacks 

camera_data = {'image': np.zeros((image_h, image_w, 3))}
#callbacks need to change
def gnss_callback(data, data_dict):
    t = world.get_snapshot().timestamp.elapsed_seconds
    true_speed = vehicle.get_velocity().length()
    heading = vehicle.get_transform().rotation.yaw
    north_off, east_off = pos_tracker.update(t, true_speed, heading)
    spoofed = apply_gps_position_bias(data, north_off, east_off)

    print(f"t={t:.3f}  offset=({north_off:.5f}, {east_off:.5f})  "
          f"real=({spoofed['original_latitude']:.7f},{spoofed['original_longitude']:.7f})  "
          f"fake=({spoofed['latitude']:.7f},{spoofed['longitude']:.7f})")

    data_dict['gnss'] = [spoofed['latitude'], spoofed['longitude']]
    
def imu_callback(data, data_dict):
    t = world.get_snapshot().timestamp.elapsed_seconds
    bias = speed_coord.extra_accel(t)   # true_speed removed — no longer needed here
    #print(f"t={t:.4f}  ratio={speed_coord.current_ratio(t):.4f}  bias={bias:.6f}")
    data_dict['imu'] = apply_imu_speed_bias(data, bias)

sensor_data = {'gnss':[0,0],
                'imu': {
                                'gyro': carla.Vector3D(),
                                'accel': carla.Vector3D(),
                                'original_accel': carla.Vector3D(),
                                'compass': 0}}

def camera_callback(image, data_dict):
    bgr = process_image(image) #this function???
    t = world.get_snapshot().timestamp.elapsed_seconds
    bgr = flow_spoofer.apply(bgr, t, dt=1.0/20)
    data_dict['image'] = bgr

In [ ]:
#listen for callbacks 
gnss.listen(lambda event: gnss_callback(event, sensor_data))
imu.listen(lambda event: imu_callback(event, sensor_data))
camera.listen(lambda image: camera_callback(image,camera_data))

t=3524.258  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3524.263  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3524.268  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)


t=3524.274  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3524.291  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3524.304  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3524.317  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3524.330  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3524.342  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3524.356  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3524.369  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3524.383  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3524.396  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3524.410  offset=(

In [ ]:
point_a = start_point.location #start where the car is

sampling_resolution = 1
grp = GlobalRoutePlanner(world.get_map(), sampling_resolution)

#pick the longest route
distance = 0
for loc in spawn_points:
    cur_route = grp.trace_route(point_a, loc.location)
    if len(cur_route)>distance:
        distance = len(cur_route)
        route = cur_route

#draw the route 
for waypoint in route:
    world.debug.draw_string(waypoint[0].transform.location, '^', draw_shadow=False,
                            color=carla.Color(r=0, g=0, b=255), life_time=60.0,
                            persistent_lines=True)

t=3530.441  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.456  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.468  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.480  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.494  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.508  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.523  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.537  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.552  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.565  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.578  offset=(

t=3530.731  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.745  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.758  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.772  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.785  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.797  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.810  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.823  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.834  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.847  offset=(0.00000, 0.00000)  real=(-0.0002198,-0.0005807)  fake=(-0.0002198,-0.0005807)
t=3530.860  offset=(

In [ ]:
for waypoint in route:

    #move the car to current waypoint
    vehicle.set_transform(waypoint[0].transform)
    # copy the latest frame so we don't draw on the camera callback's buffer
    frame = camera_data['image'].copy()

    lat, lon = sensor_data['gnss']
    gyro = sensor_data['imu']['gyro']
    accel = sensor_data['imu']['accelerometer']
    original_accel = sensor_data['imu']['original_accel']
    compass = sensor_data['imu']['compass']

    lines = [
        f"GNSS: {lat:.5f}, {lon:.5f}", #latitude and longitude in degrees
        f"Gyro: {gyro.x:.2f}, {gyro.y:.2f}, {gyro.z:.2f}", #angular velocity (how fast car is rotating around x,y,z) in radians/sec
        f"Accel: {accel.x:.2f}, {accel.y:.2f}, {accel.z:.2f}", #acceleration along x,y,z in m/s^2
        f"Original Accel: {original_accel.x:.2f}, {original_accel.y:.2f}, {original_accel.z:.2f}",
        f"Compass: {compass:.1f}",
    ]
    for i, text in enumerate(lines):
        cv2.putText(frame, text, (10, 20 + i * 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)

    cv2.imshow('Fake self-driving', frame)
    cv2.waitKey(50)

t=3534.435  offset=(-0.00017, -0.00000)  real=(-0.0002201,-0.0005823)  fake=(-0.0002201,-0.0005823)
t=3534.448  offset=(-0.00057, -0.00000)  real=(-0.0002201,-0.0005823)  fake=(-0.0002201,-0.0005823)
t=3534.463  offset=(-0.00057, -0.00000)  real=(-0.0002201,-0.0005823)  fake=(-0.0002201,-0.0005823)
t=3534.479  offset=(-0.00037, -0.00000)  real=(-0.0002201,-0.0005823)  fake=(-0.0002201,-0.0005823)
t=3534.494  offset=(-0.00026, -0.00000)  real=(-0.0002201,-0.0005731)  fake=(-0.0002201,-0.0005731)
t=3534.508  offset=(0.00002, -0.00001)  real=(-0.0002201,-0.0005731)  fake=(-0.0002201,-0.0005731)
t=3534.520  offset=(0.00078, -0.00002)  real=(-0.0002201,-0.0005731)  fake=(-0.0002201,-0.0005731)
t=3534.534  offset=(0.00062, -0.00000)  real=(-0.0002197,-0.0005632)  fake=(-0.0002197,-0.0005632)
t=3534.546  offset=(0.00145, -0.00007)  real=(-0.0002197,-0.0005632)  fake=(-0.0002197,-0.0005632)
t=3534.560  offset=(0.00290, -0.00017)  real=(-0.0002197,-0.0005632)  fake=(-0.0002197,-0.0005632)
t=353

t=3564.321  offset=(4.77975, 1.01940)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009615)
t=3564.334  offset=(4.77979, 1.01259)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009614)
t=3564.348  offset=(4.77985, 1.00381)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009613)
t=3564.362  offset=(4.77989, 0.99854)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009613)
t=3564.375  offset=(4.77991, 0.99466)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009612)
t=3564.388  offset=(4.77994, 0.99044)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009612)
t=3564.400  offset=(4.77996, 0.98863)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009612)
t=3564.413  offset=(4.77996, 0.98778)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009612)
t=3564.426  offset=(4.77996, 0.98760)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009612)
t=3564.438  offset=(4.77996, 0.98761)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009612)
t=3564.450  offset=(4.77996, 0.98754)  r

In [96]:
cv2.destroyAllWindows()
#after the car reaches the destination clear everything 
time.sleep(2)
cv2.destroyAllWindows()
camera.stop() #opposite of camera.listen()
gnss.stop()
imu.stop()
for actor in world.get_actors().filter('*vehicle*'):
    actor.destroy()
for sensor in world.get_actors().filter('*sensor*'):
    sensor.destroy()

t=3665.771  offset=(4.78018, 0.95594)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009609)
t=3665.785  offset=(4.78018, 0.95594)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009609)
t=3665.798  offset=(4.78018, 0.95594)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009609)
t=3665.812  offset=(4.78018, 0.95594)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009609)
t=3665.826  offset=(4.78018, 0.95594)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009609)
t=3665.839  offset=(4.78018, 0.95594)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009609)
t=3665.853  offset=(4.78018, 0.95594)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009609)
t=3665.865  offset=(4.78018, 0.95594)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009609)
t=3665.865  offset=(4.78018, 0.95594)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009609)
t=3665.891  offset=(4.78018, 0.95594)  real=(-0.0004695,0.0009523)  fake=(-0.0004266,0.0009609)
t=3665.904  offset=(4.78018, 0.95594)  r